# Write native UFL local forms and the global equation

The affine pressure $p=1+x+2y$ and physical Darcy flux $q=(-1,-2)$ use identity permeability and zero source. Both paths declare the actual local and global forms; the native provider supplies a mesh/integration context to the user callback.

The user supplies four blocks; the library eliminates the local complement:

$$
\begin{aligned}
A_T u_T+B_T\lambda_T &= f_T,\\
C_Tu_T+D_T\lambda_T &= g_T.
\end{aligned}
$$

Here $a_T(p,v)=(\nabla p,\nabla v)_T$, $L_T(v)=(0,v)_T$,
$B_T\lambda=\langle\lambda,v\rangle_{\partial T}$ and
$C_Tp=-\langle p,\mu\rangle_{\partial T}$ with signed macroface normals.
$D_T=0$ and $g_T=0$. The constant local kernel is retained globally;
the physical integral selects its complement. The global right-hand side is
$-\langle p_D,\mu\rangle_{\partial\Omega}$ in trace coordinates, followed
by zeros in retained-mode coordinates. No model name chooses these forms.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/7c8fc8a64205eb82cd26e787da7cac3fff8b86adcdeb445d6f06bd46a1726691/ufl_provider-companion.zip"
COMPANION_SHA256 = "7c8fc8a64205eb82cd26e787da7cac3fff8b86adcdeb445d6f06bd46a1726691"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("foundations/operators/ufl_provider.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)

import numpy as np


In [ ]:
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble, columns, rows
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.interval import SkeletonSpace
from pymhm.fem.scalar.operators import p1_operators, face_integration, boundary_data
from examples.variational_darcy import affine_pressure, DarcyProvider

mesh = TriangleMesh.unit_square()
skeleton = SkeletonSpace(mesh)

def portable_local_forms(cell):
    """Declare coefficient forms with the same P1 pressure and signed P0 trace."""
    fine = mesh.submesh(cell, 2)
    a, mass, _ = p1_operators(fine)
    b, _ = face_integration(mesh, cell, fine, skeleton)
    return LocalEquations(
        a, np.zeros(len(fine.points)), columns(*b.T), rows(*(-b.T)),
        skeleton.cell_dofs(cell), kernel=np.ones((len(fine.points), 1)),
        moments=np.asarray(mass.sum(axis=1)).reshape(-1, 1),
        metadata={"points": fine.points.copy()},
    )

boundary, _ = boundary_data(skeleton, affine_pressure)
global_equation = Equation(0, -np.r_[boundary, np.zeros(len(mesh.cells))])
problem = MultiscaleProblem(global_equation, portable_local_forms,
                            range(len(mesh.cells)), skeleton.size, (1,) * len(mesh.cells))
system = assemble(problem)
solution = system.solve()
errors = [float(np.max(np.abs(field - affine_pressure(record["points"]))))
          for field, record in zip(solution.fields, system.local_metadata, strict=True)]
print({"provider": "Basix", "pressure_nodal_error_by_cell": errors,
       "global_original_relative_residual": solution.raw_residual})
assert max(errors) < 1e-10


## Native UFL on a worker-owned local mesh

Each signed boundary integral corresponds to a declared trace coefficient. The C rows use the trial pressure, independently of the B columns. DOLFINx assembles real rank-one/two forms on COMM_SELF; native resources stay in the worker and only numerical responses reach global assembly. The global Equation here is an explicit coefficient form in reduced coordinates. A native global form requires a declared function space whose coefficient ordering matches those coordinates; cross-mesh UFL coupling is supplied through the signed local linear forms.


In [ ]:
import importlib.util

native_available = importlib.util.find_spec("dolfinx") is not None
if native_available:
    import ufl

    def user_native_forms(context):
        """Write UFL A, L, B, C and the physical pressure integral directly."""
        p, v = ufl.TrialFunction(context.space), ufl.TestFunction(context.space)
        dx = ufl.dx(domain=context.space.ufl_domain())
        return LocalEquations(
            ufl.inner(ufl.grad(p), ufl.grad(v)) * dx, 0.0 * v * dx,
            columns(*(float(sign) * v * context.ds(i + 1)
                      for i, sign in enumerate(context.signs))),
            rows(*(-float(sign) * p * context.ds(i + 1)
                   for i, sign in enumerate(context.signs))),
            context.trace_dofs, kernel=np.ones((len(context.points), 1)),
            moments=columns(v * dx),
        )

    native_provider = DarcyProvider(mesh, skeleton, "fenics", 2, user_native_forms)
    native_problem = MultiscaleProblem(
        global_equation, native_provider, range(len(mesh.cells)),
        skeleton.size, (1,) * len(mesh.cells),
    )
    native_system = assemble(native_problem)
    native_solution = native_system.solve()
    native_errors = [
        float(np.max(np.abs(field - affine_pressure(record["points"]))))
        for field, record in zip(native_solution.fields, native_system.local_metadata, strict=True)
    ]
    print({"native_available": True, "provider": "UFL/DOLFINx",
           "pressure_nodal_error_by_cell": native_errors,
           "global_original_relative_residual": native_solution.raw_residual})
    assert max(native_errors) < 1e-10
else:
    print({"native_available": False,
           "status": "Native UFL/DOLFINx not executed: select the Pixi fem kernel."})
